<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 9B: *Fire Spread Feature Ablation*
##### Version Number: 4.0
---
### Contents
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_spread = pd.read_csv('../data/processed/X_spread.csv')
y_spread = pd.read_csv('../data/processed/y_spread.csv').squeeze()  # Load as Series
details_spread = pd.read_csv('../data/processed/details_spread.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/spread_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)
    
with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_spread,y_spread], axis=1)
subset = subset_df(reform, 'Target_Spread', 500)

y = subset['Target_Spread']
X = subset.drop(columns='Target_Spread')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build tuned models
spread_xgb = xgb.XGBClassifier(**XGB_parameters)
spread_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 150,
 'max_depth': 15,
 'min_samples_split': 10,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 150,
 'max_depth': 3,
 'learning_rate': 0.4,
 'verbosity': 0}

In [7]:
models = {
    "XGB": spread_xgb, 
    "RF": spread_rf,
}

## SHAP

### XGB Spread SHAP

In [8]:
xgb_top_10 = get_shap(spread_xgb, X_xgb, y_xgb)
xgb_top_10

,0,1
0,1000-hour Dead Fuel Moisture,0.514945
1,road_density_x_forest_percent,0.495068
2,Palmer Drought Severity Index,0.436829
3,Daily Minimum Air Temperature 7 Day Avg,0.352716
4,Specific Humidity,0.323749
5,slope_max,0.291453
6,slope_std,0.288551
7,slope_mean,0.280849
8,Solar Radiation,0.268777
9,Minimum Relative Humidity 7 Day Avg,0.268416


### RF Spread SHAP

In [9]:
rf_top_10 = get_shap(spread_rf, X_rf, y_rf)
rf_top_10 

 99%|===================| 894/900 [00:37<00:00]        

,0,1
0,road_density_x_forest_percent,0.022473
1,forest_percent,0.014560
2,1000-hour Dead Fuel Moisture,0.013980
3,shrub_grass_percent,0.012848
4,slope_max,0.011107
5,Solar Radiation 7 Day Avg,0.010822
6,Season_Spring,0.010665
7,slope_mean,0.010481
8,slope_std,0.009529
9,slope_range,0.008704


## Set Ablation

In [10]:
for set_name, columns in feature_sets.items():
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['SPI 30-Day', 'SPI 180-Day', 'SPEI 30-Day', 'SPEI 90-Day', 'SPEI 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Release Component', 'Santa_Ana_Score']


Social: ['total_housing', 'total_population', 'housing_density', 'population_density', 'median_income']


Elevation: ['elevation_range', 'elevation_mean', 'elev

In [16]:
compare = []

compare.append(compare_model(spread_xgb, X, y, best_strategy,
                                     name = 'XGB', test_set = 'Full'))

compare.append(compare_model(spread_rf, X, y, best_strategy,
                                     name = 'RF', test_set = 'Full'))

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand
Testing RF: Water Demand
Testing XGB: Water Supply
Testing RF: Water Supply
Testing XGB: Water Supply Indexes
Testing RF: Water Supply Indexes
Testing XGB: Fire Danger
Testing RF: Fire Danger
Testing XGB: Social
Testing RF: Social
Testing XGB: Elevation
Testing RF: Elevation
Testing XGB: WUI
Testing RF: WUI
Testing XGB: Ecoregion
Testing RF: Ecoregion
Testing XGB: Land Cover
Testing RF: Land Cover
Testing XGB: Interactions
Testing RF: Interactions
Testing XGB: Wind Slope
Testing RF: Wind Slope
Testing XGB: Others
Testing RF: Others
Testing XGB: Coded Ecoregions
Testing RF: Coded Ecoregions
Testing XGB: Coded Seasons
Testing RF: Coded Seasons


In [17]:
comparisons = pd.DataFrame(compare)

In [18]:
XGB = comparisons[comparisons['Model'] == 'XGB'] 
RF = comparisons[comparisons['Model'] == 'RF'] 

In [19]:
full_XGB = XGB.loc[XGB['Test Set']=='Full','Macro F1'].values[0]
full_RF = RF.loc[RF['Test Set']=='Full','Macro F1'].values[0]

In [20]:
XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100

C:\Users\dusti\AppData\Local\Temp\ipykernel_8036\2524239291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
C:\Users\dusti\AppData\Local\Temp\ipykernel_8036\2524239291.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100


In [21]:
RF

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
1,Full,RF,0.756858,0.754722,0.690909,0.000000
3,Water Demand,RF,0.756454,0.755586,0.690909,-0.114477
5,Water Supply,RF,0.766979,0.763714,0.690909,-1.191320
7,Water Supply Indexes,RF,0.718615,0.719250,0.618182,4.700123
9,Fire Danger,RF,0.760561,0.757643,0.700000,-0.386942
11,Social,RF,0.750453,0.748738,0.681818,0.792922
13,Elevation,RF,0.747596,0.745444,0.690909,1.229416
15,WUI,RF,0.759982,0.757430,0.681818,-0.358777
17,Ecoregion,RF,0.750765,0.747438,0.672727,0.965122
19,Land Cover,RF,0.760906,0.759424,0.709091,-0.622953


In [22]:
XGB

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
0,Full,XGB,0.713732,0.712432,0.672727,0.000000
2,Water Demand,XGB,0.737466,0.734270,0.709091,-3.065255
4,Water Supply,XGB,0.730352,0.728566,0.681818,-2.264649
6,Water Supply Indexes,XGB,0.712503,0.715193,0.636364,-0.387508
8,Fire Danger,XGB,0.697119,0.694260,0.645455,2.550742
10,Social,XGB,0.720025,0.718849,0.663636,-0.900714
12,Elevation,XGB,0.723223,0.721668,0.654545,-1.296376
14,WUI,XGB,0.736879,0.735132,0.672727,-3.186174
16,Ecoregion,XGB,0.729384,0.728068,0.654545,-2.194692
18,Land Cover,XGB,0.729877,0.727405,0.672727,-2.101681
